# 🧠 Aula 2 — Pipeline de Processamento de Texto com spaCy

**Objetivo:** demonstrar um pipeline de PLN com foco em componentes linguísticos oferecidos pelo spaCy:
- **Tokenização**
- **Stopwords**
- **Redução morfológica (lematização)**
- (extra) **POS tagging** e **segmentação em sentenças**

> Nota: em aplicações reais, a **higienização** e parte da **normalização** continuam sendo tarefas de engenharia (regex/Unicode), enquanto o spaCy oferece os componentes linguísticos.


---

## 🧰 0) Instalação e carregamento do modelo (Português)

In [ ]:
# Instalação do spaCy (Colab)
!pip -q install spacy


In [ ]:
# Download do modelo de português
!python -m spacy download pt_core_news_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 105.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import unicodedata
import spacy
# Carrega o modelo de linguagem para português
nlp = spacy.load("pt_core_news_sm")
print("spaCy carregado. Pipeline:", nlp.pipe_names)

spaCy carregado. Pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


---

## 📌 1) Texto base

In [ ]:
texto_bruto = "Não gostei... o produto veio com defeito 😡 http://exemplo.com <p title='Destaque'>péssimo</p>"
print("Texto bruto:", texto_bruto)


Texto bruto: Não gostei... o produto veio com defeito 😡 http://exemplo.com <p title='Destaque'>péssimo</p>


---

## 🧹 2) Higienização e normalização (engenharia)

Faremos, de forma explícita:
- remoção de HTML e URLs
- remoção de símbolos/pontuação (para simplificar a análise)
- lowercase + remoção de acentos

> Observação: dependendo da tarefa, pode ser desejável manter pontuação/emojis. Nesta aula, o foco é o pipeline.


In [ ]:
def higienizar(texto: str) -> str:
    # remove HTML
    texto = re.sub(r"<[^>]+>", " ", texto)
    # remove URLs
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    # remove pontuação e símbolos (mantém letras/números/_ e espaços)
    texto = re.sub(r"[^\w\s]", " ", texto, flags=re.UNICODE)
    # normaliza espaços
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def normalizar(texto: str, remover_acentos: bool = True) -> str:
    texto = texto.lower()
    if remover_acentos:
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

texto_limpo = higienizar(texto_bruto)
texto_norm = normalizar(texto_limpo, remover_acentos=True)

print("Texto limpo:      ", texto_limpo)
print("Texto normalizado:", texto_norm)


Texto limpo:       Não gostei o produto veio com defeito péssimo
Texto normalizado: nao gostei o produto veio com defeito pessimo


---

## ✂️ 3) Tokenização com spaCy

Ao processar o texto com `nlp(texto)`, obtemos um objeto `Doc`.


In [ ]:
doc = nlp(texto_norm)
tokens = [t.text for t in doc]
print("Tokens:", tokens)

Tokens: ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito', 'pessimo']


---

## 🚫 4) Stopwords com spaCy

O spaCy marca stopwords em `token.is_stop`.

O spaCy já incorpora decisões linguísticas mais sofisticadas no modelo. A palavra *nao* não é considerada uma stopword no modelo do spaCy.

In [ ]:
def remover_stopwords_spacy(doc):
    saida = []
    for t in doc:
        # Retire o comentário para ver os stopwords
        #print(f"token:{t}\t stopword:{t.is_stop}")
        saida.append(t)
    return saida

tokens_filtrados = remover_stopwords_spacy(doc)
print("Tokens filtrados:", [t.text for t in tokens_filtrados])


Tokens filtrados: ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito', 'pessimo']


---

## 🌱 5) Redução morfológica (Lematização) e POS tagging

Cada objeto `Token` representa uma unidade linguística analisada, e não apenas uma palavra como texto:  
*   `t.text`: texto original do token, exatamente como aparece no texto processado;
*   `t.lemma_`: lema (forma canônica). É o resultado da redução morfológica por lematização;
*   `t.pos_`: classe gramatical do token (ex.: VERB, NOUN, ADJ). POS tagging refere-se a uma etapa do PLN responsável por identificar a classe gramatical de cada palavra (token) em um texto.

In [ ]:
linhas = [(t.text, t.lemma_, t.pos_) for t in tokens_filtrados]
for linha in linhas:
    print(linha)

('nao', 'nao', 'PRON')
('gostei', 'gostar', 'VERB')
('o', 'o', 'DET')
('produto', 'produto', 'NOUN')
('veio', 'vir', 'VERB')
('com', 'com', 'ADP')
('defeito', 'defeito', 'NOUN')
('pessimo', 'pessimo', 'ADJ')


---

## 🧾 6) Segmentação em sentenças

`doc.sents` é um iterador sobre as sentenças identificadas pelo modelo linguístico.

Cada elemento retornado é um objeto do tipo `Span`, que representa uma sentença completa.

In [ ]:
for i, sent in enumerate(doc.sents, 1):
    print(f"Sentença {i}: {sent.text}")


Sentença 1: nao gostei o produto veio com defeito pessimo


---

## 🧩 7) Pipeline completo

In [ ]:
def pipeline_spacy(
    texto_bruto: str,
    remover_acentos: bool = True
):
    # 1) engenharia
    texto_limpo = higienizar(texto_bruto)
    texto_norm = normalizar(texto_limpo, remover_acentos=remover_acentos)

    # 2) spaCy: tokenização + anotações
    doc = nlp(texto_norm)

    # 3) spaCy: stopwords
    tokens_filtrados = remover_stopwords_spacy(doc)

    # 4) spaCy: lematização
    lemas = [t.lemma_ for t in tokens_filtrados]

    return {
        "texto_bruto": texto_bruto,
        "texto_limpo": texto_limpo,
        "texto_normalizado": texto_norm,
        "tokens": [t.text for t in doc],
        "tokens_filtrados": [t.text for t in tokens_filtrados],
        "lemas": lemas
    }

saida = pipeline_spacy(texto_bruto, remover_acentos=True)
for k, v in saida.items():
    print(f"{k:>16}: {v}")


     texto_bruto: Não gostei... o produto veio com defeito 😡 http://exemplo.com <p title='Destaque'>péssimo</p>
     texto_limpo: Não gostei o produto veio com defeito péssimo
texto_normalizado: nao gostei o produto veio com defeito pessimo
          tokens: ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito', 'pessimo']
tokens_filtrados: ['nao', 'gostei', 'o', 'produto', 'veio', 'com', 'defeito', 'pessimo']
           lemas: ['nao', 'gostar', 'o', 'produto', 'vir', 'com', 'defeito', 'pessimo']


---

# 🧪 Exemplos

1. Rode `pipeline_spacy` para os textos abaixo e compare `tokens_filtrados` e `lemas`:
   - `"Excelente produto!!! Entrega rápida 😄 Recomendo demais!!!"`
   - `"<p>Produto ok</p>, mas a embalagem estava DANIFICADA!!!"`
   - `"Achei o produto bom, não é ruim, mas poderia ser melhor."`

In [ ]:
textos = [
    "Excelente produto!!! Entrega rápida 😄 Recomendo demais!!!",
    "<p>Produto ok</p>, mas a embalagem estava DANIFICADA!!!",
    "Achei o produto bom, não é ruim, mas poderia ser melhor."
]

for i, t in enumerate(textos, 1):
    print(f"\n--- Texto {i} ---")
    r = pipeline_spacy(t, remover_acentos=True)
    print("tokens_filtrados:", r["tokens_filtrados"])
    print("lemas          :", r["lemas"])



--- Texto 1 ---
tokens_filtrados: ['excelente', 'produto', 'entrega', 'rapida', 'recomendo', 'demais']
lemas          : ['excelente', 'produto', 'entregar', 'rapir', 'recomendo', 'demais']

--- Texto 2 ---
tokens_filtrados: ['produto', 'ok', 'mas', 'a', 'embalagem', 'estava', 'danificada']
lemas          : ['produto', 'ok', 'mas', 'o', 'embalagem', 'estar', 'danificado']

--- Texto 3 ---
tokens_filtrados: ['achei', 'o', 'produto', 'bom', 'nao', 'e', 'ruim', 'mas', 'poderia', 'ser', 'melhor']
lemas          : ['achar', 'o', 'produto', 'bom', 'nao', 'e', 'ruim', 'mas', 'poder', 'ser', 'bom']


2. Rode com `remover_acentos=False` e discuta o efeito sobre:
   - stopwords
   - negação (“não”)

In [ ]:
textos = [
    "Excelente produto!!! Entrega rápida 😄 Recomendo demais!!!",
    "<p>Produto ok</p>, mas a embalagem estava DANIFICADA!!!",
    "Achei o produto bom, não é ruim, mas poderia ser melhor."
]

for i, t in enumerate(textos, 1):
    print(f"\n--- Texto {i} ---")
    r = pipeline_spacy(t, remover_acentos=False)
    print("tokens_filtrados:", r["tokens_filtrados"])
    print("lemas          :", r["lemas"])



--- Texto 1 ---
tokens_filtrados: ['excelente', 'produto', 'entrega', 'rápida', 'recomendo', 'demais']
lemas          : ['excelente', 'produto', 'entregar', 'rápidar', 'recomer', 'demais']

--- Texto 2 ---
tokens_filtrados: ['produto', 'ok', 'mas', 'a', 'embalagem', 'estava', 'danificada']
lemas          : ['produto', 'ok', 'mas', 'o', 'embalagem', 'estar', 'danificado']

--- Texto 3 ---
tokens_filtrados: ['achei', 'o', 'produto', 'bom', 'não', 'é', 'ruim', 'mas', 'poderia', 'ser', 'melhor']
lemas          : ['achar', 'o', 'produto', 'bom', 'não', 'ser', 'ruim', 'mas', 'poder', 'ser', 'bom']
